Validation for aadt_filling_texas.ipynb outputs.

Reads the per-year CSVs in outputs/ and runs 14 checks:
schema, Crash_Year tagging, coordinate bounds, distance cap, ZIP completeness,
year_gap formula, VMT_Multiplier formula, Adt_Curnt_Amt formula, match_type consistency,
BallTree vs brute-force haversine, speed benchmark, year_gap distribution,
AADT range sanity, per-year fill rate.


In [22]:
import os, time, sys
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree


In [23]:
OUTPUTS_DIR = 'outputs'
STATIONS_PATH = 'old_aadt.csv'

EARTH_RADIUS_MILES = 3958.8
MAX_STATION_MILES  = 1.0
TX_LAT_MIN, TX_LAT_MAX = 25.8,  36.5
TX_LON_MIN, TX_LON_MAX = -106.6, -93.5
SAMPLE_SEED = 42

REQUIRED_COLS = [
    'Crash_ID', 'Latitude', 'Longitude', 'ZIP_Code',
    'Adt_Curnt_Amt', 'Distance_Miles', 'Station_Year',
    'year_gap', 'aadt_match_type', 'VMT_Multiplier',
]

TX_VMT_MILLIONS = {
    2015: 258300, 2016: 270700, 2017: 273200, 2018: 282200, 2019: 288400,
    2020: 260000, 2021: 285200, 2022: 291100, 2023: 301500, 2024: 307800,
    2025: 307800,
}

ATOL_VMT  = 1e-9
ATOL_DIST = 1e-4


In [24]:
# Discover yearly output files
year_files = []
for f in sorted(os.listdir(OUTPUTS_DIR)):
    if f.startswith('Cleaned_Data_') and f.endswith('.csv') and '_Final_' not in f:
        try:
            y = int(f.replace('Cleaned_Data_', '').replace('.csv', ''))
            year_files.append((y, os.path.join(OUTPUTS_DIR, f)))
        except ValueError:
            pass

year_files.sort()
print(f'{len(year_files)} yearly files:')
for y, p in year_files:
    print(f'  {y}: {p} ({os.path.getsize(p)/1e6:.0f} MB)')


9 yearly files:
  2016: outputs\Cleaned_Data_2016.csv (198 MB)
  2017: outputs\Cleaned_Data_2017.csv (194 MB)
  2018: outputs\Cleaned_Data_2018.csv (196 MB)
  2019: outputs\Cleaned_Data_2019.csv (200 MB)
  2020: outputs\Cleaned_Data_2020.csv (169 MB)
  2021: outputs\Cleaned_Data_2021.csv (196 MB)
  2022: outputs\Cleaned_Data_2022.csv (195 MB)
  2023: outputs\Cleaned_Data_2023.csv (149 MB)
  2024: outputs\Cleaned_Data_2024.csv (197 MB)


In [25]:
results = []

def record(name, passed, msg):
    results.append({'validation': name, 'passed': bool(passed), 'msg': msg})
    status = 'pass' if passed else 'FAIL'
    print(f'[{status}] {name}: {msg}')


V01 - schema. Every yearly file must contain the 10 columns the PDF requires.


In [26]:
for year, path in year_files:
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        header = f.readline().strip().split(',')
    missing = [c for c in REQUIRED_COLS if c not in header]
    record(
        f'V01 schema {year}',
        len(missing) == 0,
        f'missing {missing}' if missing else f'{len(REQUIRED_COLS)}/10 present, {len(header)} cols total'
    )


[pass] V01 schema 2016: 10/10 present, 92 cols total
[pass] V01 schema 2017: 10/10 present, 92 cols total
[pass] V01 schema 2018: 10/10 present, 92 cols total
[pass] V01 schema 2019: 10/10 present, 92 cols total
[pass] V01 schema 2020: 10/10 present, 92 cols total
[pass] V01 schema 2021: 10/10 present, 92 cols total
[pass] V01 schema 2022: 10/10 present, 92 cols total
[pass] V01 schema 2023: 10/10 present, 92 cols total
[pass] V01 schema 2024: 10/10 present, 92 cols total


V02 - Crash_Year tag. Every row in Cleaned_Data_YYYY.csv should have Crash_Year == YYYY.


In [27]:
for year, path in year_files:
    yrs = pd.read_csv(path, usecols=['Crash_Year'], low_memory=False)['Crash_Year'].dropna().unique()
    ok = (len(yrs) == 1) and (int(yrs[0]) == year)
    record(f'V02 Crash_Year {year}', ok, f'distinct values = {sorted(yrs.tolist())}')


[pass] V02 Crash_Year 2016: distinct values = [2016]
[pass] V02 Crash_Year 2017: distinct values = [2017]
[pass] V02 Crash_Year 2018: distinct values = [2018]
[pass] V02 Crash_Year 2019: distinct values = [2019]
[pass] V02 Crash_Year 2020: distinct values = [2020]
[pass] V02 Crash_Year 2021: distinct values = [2021]
[pass] V02 Crash_Year 2022: distinct values = [2022]
[pass] V02 Crash_Year 2023: distinct values = [2023]
[pass] V02 Crash_Year 2024: distinct values = [2024]


V03 - coordinate bounds. No row outside the TX bounding box, no null coordinates.


In [28]:
for year, path in year_files:
    df = pd.read_csv(path, usecols=['Latitude', 'Longitude'], low_memory=False)
    null_count = (df['Latitude'].isna() | df['Longitude'].isna()).sum()
    out_of_bounds = (
        (df['Latitude']  < TX_LAT_MIN) | (df['Latitude']  > TX_LAT_MAX) |
        (df['Longitude'] < TX_LON_MIN) | (df['Longitude'] > TX_LON_MAX)
    ).sum()
    record(
        f'V03 bounds {year}',
        (null_count == 0) and (out_of_bounds == 0),
        f'{null_count} null, {out_of_bounds} out-of-TX'
    )


[pass] V03 bounds 2016: 0 null, 0 out-of-TX
[pass] V03 bounds 2017: 0 null, 0 out-of-TX
[pass] V03 bounds 2018: 0 null, 0 out-of-TX
[pass] V03 bounds 2019: 0 null, 0 out-of-TX
[pass] V03 bounds 2020: 0 null, 0 out-of-TX
[pass] V03 bounds 2021: 0 null, 0 out-of-TX
[pass] V03 bounds 2022: 0 null, 0 out-of-TX
[pass] V03 bounds 2023: 0 null, 0 out-of-TX
[pass] V03 bounds 2024: 0 null, 0 out-of-TX


V04 - distance cap. No row with Distance_Miles > 1 has a populated Adt_Curnt_Amt.


In [29]:
for year, path in year_files:
    df = pd.read_csv(path, usecols=['Distance_Miles', 'Adt_Curnt_Amt'], low_memory=False)
    violations = ((df['Distance_Miles'] > MAX_STATION_MILES) & df['Adt_Curnt_Amt'].notna()).sum()
    filled_dist = df.loc[df['Adt_Curnt_Amt'].notna(), 'Distance_Miles']
    max_filled = filled_dist.max() if len(filled_dist) else float('nan')
    record(
        f'V04 dist cap {year}',
        violations == 0,
        f'{violations} violations; max Distance_Miles among filled = {max_filled:.4f}'
    )


[pass] V04 dist cap 2016: 0 violations; max Distance_Miles among filled = 1.0000
[pass] V04 dist cap 2017: 0 violations; max Distance_Miles among filled = 0.9999
[pass] V04 dist cap 2018: 0 violations; max Distance_Miles among filled = 1.0000
[pass] V04 dist cap 2019: 0 violations; max Distance_Miles among filled = 1.0000
[pass] V04 dist cap 2020: 0 violations; max Distance_Miles among filled = 1.0000
[pass] V04 dist cap 2021: 0 violations; max Distance_Miles among filled = 1.0000
[pass] V04 dist cap 2022: 0 violations; max Distance_Miles among filled = 1.0000
[pass] V04 dist cap 2023: 0 violations; max Distance_Miles among filled = 0.9999
[pass] V04 dist cap 2024: 0 violations; max Distance_Miles among filled = 1.0000


V05 - ZIP completeness. Pipeline drops rows missing ZIP, so every row should have one.


In [30]:
for year, path in year_files:
    df = pd.read_csv(path, usecols=['ZIP_Code'], low_memory=False)
    nulls = df['ZIP_Code'].isna().sum()
    record(f'V05 ZIP {year}', nulls == 0, f'{nulls} null ZIP rows')


[pass] V05 ZIP 2016: 0 null ZIP rows
[pass] V05 ZIP 2017: 0 null ZIP rows
[pass] V05 ZIP 2018: 0 null ZIP rows
[pass] V05 ZIP 2019: 0 null ZIP rows
[pass] V05 ZIP 2020: 0 null ZIP rows
[pass] V05 ZIP 2021: 0 null ZIP rows
[pass] V05 ZIP 2022: 0 null ZIP rows
[pass] V05 ZIP 2023: 0 null ZIP rows
[pass] V05 ZIP 2024: 0 null ZIP rows


V06 - year_gap formula. year_gap == abs(Crash_Year - Station_Year) for filled rows.


In [31]:
for year, path in year_files:
    df = pd.read_csv(path, usecols=['Crash_Year', 'Station_Year', 'year_gap'], low_memory=False)
    df = df.dropna(subset=['Station_Year', 'year_gap'])
    expected = (df['Crash_Year'].astype(int) - df['Station_Year'].astype(int)).abs()
    mismatches = (expected.astype(float) != df['year_gap'].astype(float)).sum()
    record(f'V06 year_gap {year}', mismatches == 0, f'{mismatches} mismatches')


[pass] V06 year_gap 2016: 0 mismatches
[pass] V06 year_gap 2017: 0 mismatches
[pass] V06 year_gap 2018: 0 mismatches
[pass] V06 year_gap 2019: 0 mismatches
[pass] V06 year_gap 2020: 0 mismatches
[pass] V06 year_gap 2021: 0 mismatches
[pass] V06 year_gap 2022: 0 mismatches
[pass] V06 year_gap 2023: 0 mismatches
[pass] V06 year_gap 2024: 0 mismatches


V07 - VMT_Multiplier formula. VMT_Multiplier == VMT[Crash_Year] / VMT[Station_Year].


In [32]:
for year, path in year_files:
    df = pd.read_csv(path, usecols=['Crash_Year', 'Station_Year', 'VMT_Multiplier'], low_memory=False)
    df = df.dropna(subset=['VMT_Multiplier', 'Station_Year'])
    df['expected'] = (
        df['Crash_Year'].astype(int).map(TX_VMT_MILLIONS) /
        df['Station_Year'].astype(int).map(TX_VMT_MILLIONS)
    )
    rel_err = ((df['expected'] - df['VMT_Multiplier']).abs() / df['expected'].abs()).fillna(0)
    mismatches = (rel_err > ATOL_VMT).sum()
    record(f'V07 VMT {year}', mismatches == 0, f'{mismatches} mismatches (rtol > {ATOL_VMT:.0e})')


[pass] V07 VMT 2016: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2017: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2018: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2019: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2020: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2021: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2022: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2023: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2024: 0 mismatches (rtol > 1e-09)


V08 - Adt_Curnt_Amt formula. Adt_Curnt_Amt == Station_Count * VMT_Multiplier.


In [33]:
for year, path in year_files:
    df = pd.read_csv(path, usecols=['Station_Count', 'VMT_Multiplier', 'Adt_Curnt_Amt'], low_memory=False)
    df = df.dropna(subset=['Adt_Curnt_Amt'])
    expected = df['Station_Count'] * df['VMT_Multiplier']
    rel_err = ((expected - df['Adt_Curnt_Amt']).abs() / df['Adt_Curnt_Amt'].abs()).fillna(0)
    mismatches = (rel_err > 1e-6).sum()
    record(f'V08 AADT {year}', mismatches == 0, f'{mismatches} mismatches (rtol > 1e-6)')


[pass] V08 AADT 2016: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2017: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2018: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2019: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2020: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2021: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2022: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2023: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2024: 0 mismatches (rtol > 1e-6)


V09 - aadt_match_type. NEAREST_STATION_VMT_NORM iff AADT filled, MISSING otherwise.


In [34]:
for year, path in year_files:
    df = pd.read_csv(path, usecols=['Adt_Curnt_Amt', 'aadt_match_type'], low_memory=False)
    bad_filled  = ((df['Adt_Curnt_Amt'].notna()) & (df['aadt_match_type'] != 'NEAREST_STATION_VMT_NORM')).sum()
    bad_missing = ((df['Adt_Curnt_Amt'].isna())  & (df['aadt_match_type'] != 'MISSING')).sum()
    record(
        f'V09 match_type {year}',
        (bad_filled == 0) and (bad_missing == 0),
        f'{bad_filled} filled+wrong-label, {bad_missing} missing+wrong-label'
    )


[pass] V09 match_type 2016: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2017: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2018: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2019: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2020: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2021: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2022: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2023: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2024: 0 filled+wrong-label, 0 missing+wrong-label


V10 - BallTree vs brute-force haversine. Rebuild the tree the same way the pipeline does,
then for 1000 random crashes find the truly nearest station by exhaustive haversine and
compare against the stored Nearest_Station_LocationId / Distance_Miles. Any mismatch
indicates a wiring bug (radians conversion, Earth radius, column choice, dedup).


In [35]:
stations = pd.read_csv(STATIONS_PATH, low_memory=False)
stations.columns = stations.columns.astype(str).str.strip()

LAT_COL, LON_COL = 'LATITUDE', 'LONGITUDE'
LOC_COL          = 'TRFC_STATN_ID'
COUNT_COL        = 'LATEST_AADT_QTY'
YEAR_COL         = 'LATEST_AADT_YR'

stations[LAT_COL]   = pd.to_numeric(stations[LAT_COL], errors='coerce')
stations[LON_COL]   = pd.to_numeric(stations[LON_COL], errors='coerce')
stations[COUNT_COL] = pd.to_numeric(stations[COUNT_COL], errors='coerce')
stations[YEAR_COL]  = pd.to_numeric(stations[YEAR_COL], errors='coerce').astype('Int64')
stations[LOC_COL]   = stations[LOC_COL].astype(str).str.strip()

stations = stations.dropna(subset=[LAT_COL, LON_COL, COUNT_COL]).copy()
stations = stations[stations[COUNT_COL] > 0].copy()
stations = (
    stations.sort_values(YEAR_COL)
            .groupby(LOC_COL, as_index=False)
            .tail(1)
            .reset_index(drop=True)
            .copy()
)
print(f'Stations after cleaning: {len(stations):,}')

stations_coords_rad = np.radians(stations[[LAT_COL, LON_COL]].values)
tree = BallTree(stations_coords_rad, metric='haversine')

stat_lat_arr = stations[LAT_COL].values
stat_lon_arr = stations[LON_COL].values
stat_id_arr  = stations[LOC_COL].values


Stations after cleaning: 100,400


In [15]:
def brute_force_nearest(crash_lat, crash_lon):
    lat1 = np.radians(crash_lat)
    lon1 = np.radians(crash_lon)
    lat2 = np.radians(stat_lat_arr)
    lon2 = np.radians(stat_lon_arr)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    d = EARTH_RADIUS_MILES * c
    idx = int(np.argmin(d))
    return stat_id_arr[idx], float(d[idx])


In [36]:
target_year, target_path = year_files[-1]
df_recent = pd.read_csv(
    target_path,
    usecols=['Latitude', 'Longitude', 'Nearest_Station_LocationId', 'Distance_Miles'],
    low_memory=False,
)
df_recent = df_recent.dropna(subset=['Latitude', 'Longitude', 'Nearest_Station_LocationId'])
sample = df_recent.sample(n=min(1000, len(df_recent)), random_state=SAMPLE_SEED).reset_index(drop=True)

id_mismatch   = 0
dist_mismatch = 0
worst_dist_err = 0.0
example_bad = []

for i, r in sample.iterrows():
    true_id, true_dist = brute_force_nearest(r['Latitude'], r['Longitude'])
    stored_id = str(r['Nearest_Station_LocationId']).strip()
    if str(true_id) != stored_id:
        id_mismatch += 1
        if len(example_bad) < 5:
            example_bad.append((stored_id, str(true_id), r['Distance_Miles'], true_dist))
    dist_err = abs(true_dist - float(r['Distance_Miles']))
    worst_dist_err = max(worst_dist_err, dist_err)
    if dist_err > ATOL_DIST:
        dist_mismatch += 1

ok = (id_mismatch == 0) and (dist_mismatch == 0)
record(
    f'V10 BallTree vs brute-force (n={len(sample)}, year={target_year})',
    ok,
    f'{id_mismatch} ID mismatches, {dist_mismatch} distance mismatches (max diff {worst_dist_err:.2e} mi)'
)

if not ok:
    print('First mismatches (stored_id, brute_id, stored_dist, brute_dist):')
    for row in example_bad:
        print(f'  {row}')


[pass] V10 BallTree vs brute-force (n=1000, year=2024): 0 ID mismatches, 0 distance mismatches (max diff 1.11e-16 mi)


V11 - speed benchmark. Times BallTree vs brute-force on 10,000 crashes. Informational.


In [37]:
bench_n = min(10_000, len(df_recent))
bench = df_recent.sample(n=bench_n, random_state=SAMPLE_SEED).reset_index(drop=True)
crash_coords = bench[['Latitude', 'Longitude']].values

t0 = time.perf_counter()
crash_rad = np.radians(crash_coords)
distances_rad, indices = tree.query(crash_rad, k=1)
t_balltree = time.perf_counter() - t0

t0 = time.perf_counter()
chunk = 500
brute_indices = np.empty(bench_n, dtype=np.int64)
stat_lat_r = np.radians(stat_lat_arr)
stat_lon_r = np.radians(stat_lon_arr)
for start in range(0, bench_n, chunk):
    end = min(start + chunk, bench_n)
    lat1 = np.radians(crash_coords[start:end, 0])[:, None]
    lon1 = np.radians(crash_coords[start:end, 1])[:, None]
    dlat = stat_lat_r[None, :] - lat1
    dlon = stat_lon_r[None, :] - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(stat_lat_r[None, :]) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    d = EARTH_RADIUS_MILES * c
    brute_indices[start:end] = np.argmin(d, axis=1)
t_brute = time.perf_counter() - t0

speedup = t_brute / t_balltree if t_balltree > 0 else float('inf')
record(
    'V11 speed',
    True,
    f'BallTree {t_balltree:.3f}s, brute-force {t_brute:.3f}s, {speedup:.1f}x speedup'
)

agree = int((indices.flatten() == brute_indices).sum())
print(f'BallTree/brute-force index agreement: {agree:,}/{bench_n:,}')


[pass] V11 speed: BallTree 0.851s, brute-force 73.635s, 86.5x speedup
BallTree/brute-force index agreement: 10,000/10,000


V12 - year_gap distribution. Median, p90, max per year.


In [18]:
for year, path in year_files:
    df = pd.read_csv(path, usecols=['year_gap'], low_memory=False).dropna()
    if len(df) == 0:
        record(f'V12 year_gap dist {year}', False, 'no filled rows')
        continue
    g = df['year_gap'].astype(int)
    record(
        f'V12 year_gap dist {year}',
        True,
        f'median={g.median():.0f}, p90={g.quantile(0.9):.0f}, max={g.max():d}, n={len(g):,}'
    )


[pass] V12 year_gap dist 2016: median=6, p90=7, max=12, n=970,938
[pass] V12 year_gap dist 2017: median=5, p90=6, max=13, n=472,854
[pass] V12 year_gap dist 2018: median=4, p90=5, max=14, n=478,170
[pass] V12 year_gap dist 2019: median=3, p90=4, max=15, n=492,698
[pass] V12 year_gap dist 2020: median=2, p90=3, max=16, n=412,779
[pass] V12 year_gap dist 2021: median=2, p90=2, max=17, n=479,606
[pass] V12 year_gap dist 2022: median=1, p90=3, max=18, n=483,860
[pass] V12 year_gap dist 2023: median=1, p90=4, max=19, n=367,744
[pass] V12 year_gap dist 2024: median=2, p90=5, max=20, n=475,897


V13 - AADT range. Flags filled values outside [10, 500000] above 1% of rows.


In [19]:
for year, path in year_files:
    df = pd.read_csv(path, usecols=['Adt_Curnt_Amt'], low_memory=False).dropna()
    if len(df) == 0:
        record(f'V13 AADT range {year}', False, 'no filled rows')
        continue
    v = df['Adt_Curnt_Amt']
    p5  = v.quantile(0.05)
    p95 = v.quantile(0.95)
    median = v.median()
    too_low  = (v < 10).sum()
    too_high = (v > 500_000).sum()
    ok = (too_low < len(v) * 0.01) and (too_high < len(v) * 0.01)
    record(
        f'V13 AADT range {year}',
        ok,
        f'median={median:,.0f}, p5={p5:,.0f}, p95={p95:,.0f}, <10: {too_low}, >500k: {too_high}'
    )


[pass] V13 AADT range 2016: median=6,786, p5=220, p95=37,329, <10: 1200, >500k: 0
[pass] V13 AADT range 2017: median=6,818, p5=214, p95=37,732, <10: 660, >500k: 0
[pass] V13 AADT range 2018: median=7,042, p5=221, p95=39,340, <10: 681, >500k: 0
[pass] V13 AADT range 2019: median=7,325, p5=233, p95=40,162, <10: 668, >500k: 0
[pass] V13 AADT range 2020: median=6,280, p5=192, p95=35,853, <10: 686, >500k: 0
[pass] V13 AADT range 2021: median=7,151, p5=231, p95=39,716, <10: 721, >500k: 0
[pass] V13 AADT range 2022: median=7,362, p5=240, p95=40,395, <10: 627, >500k: 0
[pass] V13 AADT range 2023: median=7,646, p5=247, p95=41,978, <10: 487, >500k: 0
[pass] V13 AADT range 2024: median=7,859, p5=250, p95=43,518, <10: 651, >500k: 0


V14 - fill rate per year.


In [20]:
total_rows = 0
total_filled = 0
for year, path in year_files:
    df = pd.read_csv(path, usecols=['Adt_Curnt_Amt'], low_memory=False)
    total = len(df)
    filled = int(df['Adt_Curnt_Amt'].notna().sum())
    pct = 100.0 * filled / total if total > 0 else 0.0
    total_rows += total
    total_filled += filled
    record(f'V14 fill rate {year}', True, f'{filled:,}/{total:,} ({pct:.2f}%)')

overall = 100.0 * total_filled / total_rows if total_rows > 0 else 0.0
print(f'all years: {total_filled:,}/{total_rows:,} = {overall:.2f}%')


[pass] V14 fill rate 2016: 957,124/994,378 (96.25%)
[pass] V14 fill rate 2017: 466,324/484,825 (96.18%)
[pass] V14 fill rate 2018: 471,780/490,927 (96.10%)
[pass] V14 fill rate 2019: 485,867/504,885 (96.23%)
[pass] V14 fill rate 2020: 407,145/423,987 (96.03%)
[pass] V14 fill rate 2021: 472,915/491,698 (96.18%)
[pass] V14 fill rate 2022: 477,165/496,021 (96.20%)
[pass] V14 fill rate 2023: 362,885/377,107 (96.23%)
[pass] V14 fill rate 2024: 469,788/488,479 (96.17%)
all years: 4,570,993/4,752,307 = 96.18%


Summary.


In [21]:
summary = pd.DataFrame(results)
summary['status'] = summary['passed'].map({True: 'pass', False: 'FAIL'})
summary = summary[['status', 'validation', 'passed', 'msg']]

n_pass = int(summary['passed'].sum())
n_fail = int((~summary['passed']).sum())

print(f'{n_pass} pass, {n_fail} fail (of {len(summary)})')
with pd.option_context('display.max_colwidth', None, 'display.width', None):
    print(summary.to_string(index=False))

summary.to_csv(os.path.join(OUTPUTS_DIR, 'validation_results.csv'), index=False)
print(f'saved {os.path.join(OUTPUTS_DIR, "validation_results.csv")}')


110 pass, 0 fail (of 110)
status                                      validation  passed                                                           msg
  pass                                 V01 schema 2016    True                                  10/10 present, 92 cols total
  pass                                 V01 schema 2017    True                                  10/10 present, 92 cols total
  pass                                 V01 schema 2018    True                                  10/10 present, 92 cols total
  pass                                 V01 schema 2019    True                                  10/10 present, 92 cols total
  pass                                 V01 schema 2020    True                                  10/10 present, 92 cols total
  pass                                 V01 schema 2021    True                                  10/10 present, 92 cols total
  pass                                 V01 schema 2022    True                                  10/